# 1. Stiffness, Eigenvalues

In [ ]:
from main import *

#####Given Functions#####
def forward_euler(f, u_start, t_end, dt):
    """March u' = f(u, t) from t = 0 to t_end with forward Euler (Lecture 10).

    Returns
    -------
    u : ndarray, shape (n + 1, len(u_start)); row k is the state at t_k
    times : ndarray, the n + 1 times t_k
    """
    n = int(round(t_end / dt))
    u = np.empty((n + 1, np.size(u_start)))
    u[0] = u_start
    for k in range(n):
        u[k + 1] = u[k] + dt * f(u[k], k * dt)
    return u, np.arange(n + 1) * dt

# Object

In [ ]:
Ar = 1.8e13 / np.sqrt(1000.0)
Ea = 17614.0 * 8.314e3
tube = chamber_solver(5, 3e6, 0.3, 1, Ar, Ea)

# Forward Euler & Stifness Visualization

In [ ]:
E, Rs, dx_lim = tube.stability_along_solution()

s0, R, gamma, h1, h2, h3 = tube.initial_conditions()

f_fn = lambda *u: tube.rhs(u, R, gamma, h1, h2, h3)

dx_small, dx_large, dx_ref = 0.99*dx_lim, 1.01*dx_lim, 0.03*dx_lim
L = 1e-4

rhs_tube = lambda u, x: np.array(f_fn(*u), float).ravel()

fig, (ax, axY2, axT, axR, axV) = plt.subplots(5, 1, figsize=(8, 17), sharex=True)
for dx, ls, lab in [(dx_small, ':', 'below limit'), (dx_large, '--', 'above limit'), (dx_ref, '-', 'reference')]:
    U, x = forward_euler(rhs_tube, s0, L, dx)
    lbl = f"dx = {dx:.1e} m ({lab})"
    ax.plot(x*1e6,   U[:, 3], ls, label=lbl)
    axY2.plot(x*1e6, U[:, 4], ls, label=lbl)
    axT.plot(x*1e6,  U[:, 1], ls, label=lbl)
    axR.plot(x*1e6,  U[:, 0], ls, label=lbl)
    axV.plot(x*1e6,  U[:, 2], ls, label=lbl)
    if lab == 'reference':
        x_ref, U_ref = x, U

xr  = x_ref * 1e6
Y1r = U_ref[:, 3]
x_burn = xr[np.argmax(Y1r < 0.5)]
x_end  = xr[np.argmax(Y1r < 1e-3)]
pad = max(x_end - x_burn, 0.05 * x_burn)

ax.set_ylabel("Y1")
ax.set_ylim(-0.1, 1.1)
ax.legend(); ax.grid(alpha=0.3)

axY2.set_ylabel("Y2")
axY2.set_ylim(-0.1, 1.1)
axY2.grid(alpha=0.3)

axT.set_ylabel("T (K)")
axT.set_ylim(0, 6000)
axT.grid(alpha=0.3)

axR.set_ylabel("ρ (kg/m³)")
axR.grid(alpha=0.3)

axV.set_ylabel("V (m/s)")
axV.set_xlabel("x (µm)")
axV.grid(alpha=0.3)

ax.set_xlim(max(x_burn - 3*pad, 0), x_end + 5*pad)
plt.tight_layout(); plt.show()